## Final Project

In [1]:
from pynq.overlays.base import BaseOverlay
import time
from datetime import datetime
import multiprocessing
import socket
import os
from pynq.lib.arduino import Arduino_Analog
base = BaseOverlay("base.bit")
btns = base.btns_gpio
leds = base.leds
from pynq.lib.arduino import Grove_OLED, ARDUINO_GROVE_I2C
import numpy
import cairo
import math
import random as rd
import socket
import os
import json
import select

## Reset GPIO on MicroBlaze and Initialize Read/Write on PMOD Pins

In [2]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn off all GPIO pins on a PMOD
void reset_gpio_b(){
    for (unsigned int i = 0; i < 8; i++)
    {
        gpio pin_out = gpio_open(i);
        gpio_set_direction(pin_out, GPIO_OUT);
        gpio_write(pin_out, 0);
    }
}

In [3]:
reset_gpio_b()

In [4]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
unsigned int write_gpio_b(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

//Function to read the value of a selected pin of PMODB
unsigned int read_gpio_b(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}

In [5]:
%%microblaze base.PMODA

#include "gpio.h"
#include "pyprintf.h"

//Function to turn off all GPIO pins on a PMOD
void reset_gpio_a(){
    for (unsigned int i = 0; i < 8; i++)
    {
        gpio pin_out = gpio_open(i);
        gpio_set_direction(pin_out, GPIO_OUT);
        gpio_write(pin_out, 0);
    }
}

In [6]:
reset_gpio_a()

In [7]:
%%microblaze base.PMODA

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
unsigned int write_gpio_a(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

//Function to read the value of a selected pin of PMODB
unsigned int read_gpio_a(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}

In [8]:
def map_val(value, fromLow, fromHigh, toLow, toHigh):
    return (value - fromLow) * (toHigh - toLow) / (fromHigh - fromLow) + toLow

In [9]:
def direction(x, y):
        flag = 0
        if x > 0.55:
            flag = 1
        if x < -.55:
            flag = 2
        if y > .55:
            flag = 3
        if y < -.55:
            flag = 4
        return flag

In [10]:
w = 64 # map will always be a square, so only one dimension is needed

In [11]:
# Source - https://stackoverflow.com/a/10031877
# Posted by Sven Marnach, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-08, License - CC BY-SA 3.0

def beep(ct, s):
    i = 0
    while i < ct:
        write_gpio_a(3,1)
        time.sleep(s)
        write_gpio_a(3,0)
        time.sleep(s)
        i = i + 1


def map_generation(width, coord):


    data = numpy.zeros((width, width, 4), dtype=numpy.uint8) # data numpy object for generating comparisons between current data & color
    surface = cairo.ImageSurface.create_for_data(
        data, cairo.FORMAT_ARGB32, width, width)
    cr = cairo.Context(surface) # circle used in png

    # fill with solid white

    cr.set_source_rgb(1.0, 1.0, 1.0) # white zone
    cr.paint()

    # draw green radius (outer zone)
    print("Generating green zone...")
    cr.arc(coord[0], coord[1], width/4, 0, 2*math.pi) #arc(xcoord, ycoord, radius, ??, circle) 
    cr.set_source_rgb(0.0, 1.0, 0.0) #green part of circle
    cr.fill()

    for i in range(width):
        for j in range(width):
            if data[i,j,0] < 128:
                data[i,j,0] = 128

    # draw yellow radius (middle zone)
    print("Generating yellow zone...")
    cr.arc(coord[0], coord[1], width/7, 0, 2*math.pi) 
    cr.set_source_rgb(1.0, 1.0, 0.0) #yellow part of circle
    cr.fill()

    for i in range(width):
        for j in range(width):
            if data[i,j,0] < 64:
                data[i,j,0] = 64

    # draw red radius (inner zone)
    print("Generating red zone...")
    cr.arc(coord[0], coord[1], width/25, 0, 2*math.pi) 
    cr.set_source_rgb(1.0, 0.0, 0.0) #red part of circle
    cr.fill()

    for i in range(width):
        for j in range(width):
            if data[i,j,0] < 32:
                data[i,j,0] = 32

    # draw dig radius
    print("Generating dig zone...")
    cr.arc(coord[0], coord[1], 2, 0, 2*math.pi) 
    cr.set_source_rgb(0.0, 0.0, 0.0) # black
    cr.fill()

    # write output
    #print(coord)
    #print(data[(coord[1]-8):(coord[1]+8), (coord[0]-7):(coord[0]+7), 0])
    surface.write_to_png("circle_complete_2.png")
    
    return data

def gameLoop(game_conn):
    analog_stick = Arduino_Analog(base.ARDUINO, [0, 1])
    write_gpio_b(1,0)
    write_gpio_b(2,0)
    write_gpio_b(3,0)
    
    y = 10
    x = 10
    prox = "null"
    win = False
    lose = False
    blink = False

    max_x = 63
    max_y = 63
    
    data = game_conn.recv()
    
    while win == False and lose == False:

        vals = analog_stick.read() 

        x_val = vals[0]
        y_val = vals[1]

        y_val_map = map_val(y_val, 0, 3.3, 1, -1)
        x_val_map = map_val(x_val, 0, 3.3, -1, 1)

        y_val_map = round(y_val_map, 2)
        x_val_map = round(x_val_map, 2)

        direc = direction(x_val_map, y_val_map)

        coordinatex = 100
        coordinatey = 100

        if direc == 1:
            x = x + 1
            time.sleep(.1)
        if direc == 2:
            x = x - 1
            time.sleep(.1)
        if direc == 3:
            y = y - 1
            time.sleep(.1)
        if direc == 4:
            y = y + 1
            time.sleep(.1)

        x = max(0,min(x,max_x))
        y = max(0,min(y,max_y))

        if data[y,x,0] == 255:
            prox = "BLANK "
            write_gpio_b(3,0)
            write_gpio_b(2,0)

        if data[y,x,0] < 255 and data[y,x,0] > 127:
            prox = "GREEN "
            write_gpio_b(3,1)
            write_gpio_b(2,0)

        if data[y,x,0] < 128 and data[y,x,0] > 63:
            prox = "YELLOW"
            write_gpio_b(3,1)
            write_gpio_b(2,1)

        if data[y,x,0] < 64 and data[y,x,0] > 31:
            prox = "RED   "
            write_gpio_b(3,0)
            write_gpio_b(2,1)

        if data[y,x,0] < 32:
            prox = "DIG   "
            blink = not blink
            if blink:
                write_gpio_b(2,1)
            if not blink:
                write_gpio_b(2,0)
            if (btns[0].read() == 1):
                win = True
                msg = '1'
                b_en = msg.encode(encoding= "utf-8")

        if game_conn.poll():
            loss_sig = game_conn.recv()
            lose = True

        print(f"X: {x:2d} Y: {y:2d}", end="\r")
        time.sleep(.1)
            
    if win == True:
        game_conn.send("0")
        winstr = "You found the treasure!!"
        print("\n")
        print(winstr.center(40,'#'))
        print("\n")
        write_gpio_b(2,0)
        for i in range(20):
            write_gpio_b(1,1)
            time.sleep(1)
            write_gpio_b(1,0)
            time.sleep(1)
    else:
        losestr = "You lost! Game over."
        print("\n")
        print(losestr.center(40,'-'))
        print("\n")
        freq = 4500
        slp = 1/(2*freq)
        count = 700
        beep(count,slp)
        write_gpio_b(1,0)
        write_gpio_b(2,0)
        write_gpio_b(3,0)
        for i in range(20):
            write_gpio_b(2,1)
            time.sleep(1)
            write_gpio_b(2,0)
            time.sleep(1)

In [12]:
def clientProc(net_conn):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.bind(('192.168.8.126',12345))
    sock.listen()
    print("Waiting for connection...")
    (clientSock,addr) = sock.accept()
    print("Connection received!")
    rec = clientSock.recv(1024)
    if rec:
        #print(type(rec))
        #print(f'data received: {rec}')
        clientSock.sendall(b'ACK: message sent successfully')
    print("Coordinates received.")
    c = json.loads(rec)
    d = map_generation(w,c)
    net_conn.send(d)
    flag = True
    while flag:
        check, g, t = select.select([clientSock, net_conn], [], [])
        for look in check:
            if look == clientSock:
                loss_msg = clientSock.recv(1024)
                deco = loss_msg.decode()
                net_conn.send(deco)
            elif look == net_conn:
                win_msg = net_conn.recv()
                byte_encode = win_msg.encode(encoding= "utf-8")
                clientSock.send(byte_encode)
        flag = False
    sock.close()
    time.sleep(5)
    print("\nClient closed!")

In [14]:
procs = [] # a future list of all our processes

game_conn, net_conn = multiprocessing.Pipe()

# Launch process1
p1_start = time.time()
p1 = multiprocessing.Process(target=clientProc, args=(net_conn,))
p1.start() # start the process
print("Server started!")
procs.append(p1)

# Launch process2
p2_start = time.time()
p2 = multiprocessing.Process(target=gameLoop, args=(game_conn,))
p2.start() # start the process
procs.append(p2)

Server started!
Waiting for connection...
Connection received!
Coordinates received.
Generating green zone...
Generating yellow zone...
Generating red zone...
Generating dig zone...
X: 58 Y: 37

########You found the treasure!!########



Client closed!
